# Ecommerce EDA Analysis

This notebook is a deeper exploratory analysis pass for an ecommerce dataset.

It is designed to help with:
- dataset loading and schema validation
- missing data, duplicates, and column quality checks
- numeric, categorical, and datetime exploration
- KPI discovery for orders, customers, products, and revenue
- trend and segment analysis for business reporting

## 1. Setup

If you have multiple candidate datasets, set `DATA_FILE` to a relative path such as `data/raw/orders.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_SEARCH_DIRS = [PROJECT_ROOT / "data" / "raw", PROJECT_ROOT / "data" / "processed"]
SUPPORTED_SUFFIXES = {".csv", ".parquet"}
DATA_FILE = None


In [ ]:
candidate_files = []
for data_dir in DATA_SEARCH_DIRS:
    if data_dir.exists():
        candidate_files.extend(
            path for path in sorted(data_dir.iterdir()) if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
        )

if DATA_FILE is None:
    if len(candidate_files) == 1:
        data_path = candidate_files[0]
    elif len(candidate_files) > 1:
        raise ValueError(
            "Multiple data files found. Set DATA_FILE to one of: " + ", ".join(path.as_posix() for path in candidate_files)
        )
    else:
        raise FileNotFoundError(
            "No CSV or Parquet files found in data/raw or data/processed. Add a dataset or set DATA_FILE manually."
        )
else:
    data_path = Path(DATA_FILE)
    if not data_path.is_absolute():
        data_path = PROJECT_ROOT / DATA_FILE

data_path


In [ ]:
if data_path.suffix.lower() == ".csv":
    df = pd.read_csv(data_path)
elif data_path.suffix.lower() == ".parquet":
    df = pd.read_parquet(data_path)
else:
    raise ValueError(f"Unsupported file format: {data_path.suffix}")

df.columns = [col.strip() for col in df.columns]

print(f"Loaded: {data_path}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()


## 2. Helper Functions

In [ ]:
def find_columns_by_keywords(columns, keywords):
    matches = []
    for col in columns:
        lowered = col.lower()
        if any(keyword in lowered for keyword in keywords):
            matches.append(col)
    return matches


def first_existing(columns, keywords):
    matches = find_columns_by_keywords(columns, keywords)
    return matches[0] if matches else None


def plot_top_categories(dataframe, column, top_n=10, title_prefix="Top Categories"):
    counts = dataframe[column].fillna("<missing>").astype(str).value_counts().head(top_n)
    plt.figure(figsize=(12, 4))
    sns.barplot(x=counts.values, y=counts.index, palette="mako")
    plt.title(f"{title_prefix}: {column}")
    plt.xlabel("Count")
    plt.ylabel(column)
    plt.show()


def safe_to_datetime(series, threshold=0.7):
    converted = pd.to_datetime(series, errors="coerce")
    return converted if converted.notna().mean() >= threshold else None


## 3. Data Overview

In [ ]:
schema_summary = pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null_count": df.notna().sum().values,
        "null_count": df.isna().sum().values,
        "null_pct": (df.isna().mean() * 100).round(2).values,
        "unique_values": df.nunique(dropna=True).values,
    }
).sort_values(["null_pct", "unique_values"], ascending=[False, False])

schema_summary


In [ ]:
overview = pd.DataFrame(
    {
        "metric": ["rows", "columns", "duplicate_rows", "duplicate_row_pct"],
        "value": [
            len(df),
            df.shape[1],
            int(df.duplicated().sum()),
            round(df.duplicated().mean() * 100, 2) if len(df) else 0,
        ],
    }
)

overview


In [ ]:
missing_values = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)
missing_values["missing_pct"] = ((missing_values["missing_count"] / len(df)) * 100).round(2)
missing_values = missing_values[missing_values["missing_count"] > 0]

missing_values if not missing_values.empty else "No missing values detected."


In [ ]:
if not missing_values.empty:
    plt.figure(figsize=(12, max(4, len(missing_values) * 0.35)))
    sns.barplot(
        data=missing_values.reset_index().rename(columns={"index": "column"}),
        x="missing_pct",
        y="column",
        palette="crest",
    )
    plt.title("Missing Values by Column")
    plt.xlabel("Missing Percentage")
    plt.ylabel("Column")
    plt.show()
else:
    print("No missing-value chart generated because the dataset is complete.")


## 4. Column-Type Exploration

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

datetime_candidates = find_columns_by_keywords(
    df.columns,
    ["date", "time", "timestamp", "created", "updated", "order", "purchase"],
)

converted_datetime_cols = []
for col in datetime_candidates:
    converted = safe_to_datetime(df[col])
    if converted is not None:
        df[col] = converted
        converted_datetime_cols.append(col)

pd.DataFrame(
    {
        "type": ["numeric", "categorical", "datetime_like"],
        "count": [len(numeric_cols), len(categorical_cols), len(converted_datetime_cols)],
        "columns": [numeric_cols, categorical_cols, converted_datetime_cols],
    }
)


In [ ]:
if numeric_cols:
    numeric_summary = df[numeric_cols].describe().T
    numeric_summary["missing_pct"] = (df[numeric_cols].isna().mean() * 100).round(2)
    numeric_summary["zero_pct"] = ((df[numeric_cols] == 0).mean() * 100).round(2)
    numeric_summary["skew"] = df[numeric_cols].skew(numeric_only=True).round(2)
    numeric_summary.sort_values("std", ascending=False)
else:
    print("No numeric columns found.")


In [ ]:
if numeric_cols:
    for column in numeric_cols[:6]:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        sns.histplot(df[column].dropna(), kde=True, ax=axes[0], color="#2a9d8f")
        axes[0].set_title(f"Distribution: {column}")
        sns.boxplot(x=df[column], ax=axes[1], color="#e9c46a")
        axes[1].set_title(f"Boxplot: {column}")
        plt.tight_layout()
        plt.show()
else:
    print("Skipping numeric charts because no numeric columns were found.")


In [ ]:
if categorical_cols:
    categorical_summary = pd.DataFrame(
        {
            "column": categorical_cols,
            "unique_values": [df[col].nunique(dropna=True) for col in categorical_cols],
            "missing_pct": [(df[col].isna().mean() * 100).round(2) for col in categorical_cols],
            "top_value": [df[col].mode(dropna=True).iat[0] if not df[col].mode(dropna=True).empty else np.nan for col in categorical_cols],
            "top_value_freq": [df[col].value_counts(dropna=True).iloc[0] if not df[col].value_counts(dropna=True).empty else 0 for col in categorical_cols],
        }
    ).sort_values(["unique_values", "missing_pct"], ascending=[False, False])
    categorical_summary
else:
    print("No categorical columns found.")


In [ ]:
if categorical_cols:
    for column in categorical_cols[:5]:
        plot_top_categories(df, column)
else:
    print("Skipping categorical charts because no categorical columns were found.")


## 5. KPI Discovery

In [ ]:
order_col = first_existing(df.columns, ["order_id", "order", "invoice", "transaction"])
customer_col = first_existing(df.columns, ["customer_id", "customer", "user_id", "user"])
product_col = first_existing(df.columns, ["product_id", "product", "sku", "item"])
category_col = first_existing(df.columns, ["category", "segment", "department", "brand"])
region_col = first_existing(df.columns, ["country", "state", "city", "region", "market"])
quantity_col = first_existing(df.columns, ["quantity", "qty", "units"])
revenue_col = first_existing(df.columns, ["sales", "revenue", "amount", "total", "price"])
date_col = first_existing(converted_datetime_cols, ["date", "time", "timestamp", "created", "updated", "order", "purchase"])

detected_fields = pd.DataFrame(
    {
        "business_field": ["order", "customer", "product", "category", "region", "quantity", "revenue", "date"],
        "column": [order_col, customer_col, product_col, category_col, region_col, quantity_col, revenue_col, date_col],
    }
)

detected_fields


In [ ]:
kpis = {
    "rows": len(df),
    "columns": df.shape[1],
    "unique_orders": df[order_col].nunique(dropna=True) if order_col else np.nan,
    "unique_customers": df[customer_col].nunique(dropna=True) if customer_col else np.nan,
    "unique_products": df[product_col].nunique(dropna=True) if product_col else np.nan,
    "total_quantity": df[quantity_col].sum() if quantity_col else np.nan,
    "total_revenue": df[revenue_col].sum() if revenue_col else np.nan,
    "avg_order_value": (
        df.groupby(order_col)[revenue_col].sum().mean() if order_col and revenue_col else np.nan
    ),
}

pd.DataFrame({"metric": list(kpis.keys()), "value": list(kpis.values())})


## 6. Trend and Segment Analysis

In [ ]:
if date_col:
    time_grain = "D"
    trend_df = df.dropna(subset=[date_col]).copy()
    trend_df["period"] = trend_df[date_col].dt.to_period(time_grain).dt.to_timestamp()

    agg_map = {"records": (date_col, "size")}
    if revenue_col:
        agg_map["revenue"] = (revenue_col, "sum")
    if quantity_col:
        agg_map["quantity"] = (quantity_col, "sum")

    time_summary = trend_df.groupby("period").agg(**agg_map).reset_index()
    time_summary.head()
else:
    print("No reliable datetime column was detected for time-series analysis.")


In [ ]:
if date_col:
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=time_summary, x="period", y="records", marker="o", label="Records")
    if "revenue" in time_summary.columns:
        sns.lineplot(data=time_summary, x="period", y="revenue", marker="o", label="Revenue")
    plt.title(f"Trend Over Time Based on {date_col}")
    plt.xlabel("Period")
    plt.ylabel("Value")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping trend chart because no datetime column was available.")


In [ ]:
segment_col = category_col or region_col or customer_col or product_col

if segment_col and revenue_col:
    segment_summary = (
        df.groupby(segment_col, dropna=False)[revenue_col]
        .agg(["sum", "mean", "count"])
        .sort_values("sum", ascending=False)
        .head(15)
        .reset_index()
    )
    segment_summary
else:
    print("Need at least one segment column and one revenue-like column for segment analysis.")


In [ ]:
if segment_col and revenue_col:
    plt.figure(figsize=(12, 5))
    sns.barplot(data=segment_summary, x="sum", y=segment_col, palette="viridis")
    plt.title(f"Top Segments by Revenue: {segment_col}")
    plt.xlabel("Revenue")
    plt.ylabel(segment_col)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping segment chart because the required columns were not detected.")


In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
    plt.title("Correlation Heatmap")
    plt.show()
else:
    print("Need at least two numeric columns to compute correlations.")


## 7. Quick Takeaways

Use this section after running the notebook to capture the main findings.

- Which metrics look like the strongest business KPIs?
- Which columns need cleaning before modeling or dashboarding?
- Are there notable seasonality, customer, product, or region patterns?
- Which segment appears to drive the highest value?